# Demo: Multifidelity UQ for Facility Location Problem 

This demo shows how to run the ACV-MRP algorithm (multifidelity) on the discrete facility-location problem with a given first-stage candidate solution.

The output is a point estimate and confidence interval for an upper bound on the optimality gap, using a low-fidelity control variate.

In [1]:
from sparow.conf_intervals.options import UQOptions
from sparow.conf_intervals.acv_mrp import ACVMRP
from sparow.conf_intervals.evaluate_true_optimality_gap import TrueOptimalityGapEvaluator

# This is the function you have to define for your problem instance
from sparow_examples.mrp_facilityloc.mrp_discrete_facilityloc import get_model_ensemble_for_uq

[    0.00] Initializing mpi-sppy
Alternative solutions package from or_topas is available.


In [ ]:
# STEP 1 - Get the multifidelity model ensemble
ensemble = get_model_ensemble_for_uq(
    model_name="HF",
    use_integer=False,
    seed=12345,
    with_replacement=True,
    lf_model_type="", # dummy argument here since facilityloc has only one LF model to choose from
)

hf_model = ensemble.high_fidelity_model()
lf_model = ensemble.low_fidelity_model()

# STEP 2 - Define the first-stage candidate solution (xhat) to evaluate
# Here, xhat specifies which facilities are opened.
xhat = {
    "x[0]": 0.0,
    "x[1]": 0.0,
    "x[2]": 0.0,
    "x[3]": 0.0,
    "x[4]": 1.0,
    "x[5]": 1.0,
}

# STEP 3 - Define the ACV-MRP options
options = UQOptions(
    n=100,            # number of scenarios per replication batch
    m=30,             # number of paired HF/LF replications
    M=10,             # number of additional LF-only replications
    alpha=0.05,       # significance level for the confidence interval
    seed=12345,       # random seed for reproducibility of algorithm run
    with_replacement=True,
    solver_name="gurobi_direct",
    verbose=True,
)

# STEP 4 - Run the ACV-MRP algorithm
acv_algorithm = ACVMRP(
    hf_model=hf_model,
    lf_model=lf_model,
    options=options,
)

results = acv_algorithm.run(xhat=xhat)

print("\nACV-MRP results:")
print(f"ACV-MRP Point Estimate: {results['point_estimate']}")
print(f"ACV-MRP Confidence Interval: [{results['ci_lower']}, {results['ci_upper']}]")
print(f"Estimated control variate coefficient: {results['control_variate_coefficient']}")
print(f"Estimated sample correlation: {results['sample_correlation']}")
print("\n")
print(f"Point estimate using only high-fidelity model (no paired or extra low-fidelity evals): {results['point_estimate_hf_only']}")
print(f"Variance reduction factor from spending additional computation on low-fidelity evals: {results['variance_reduction_factor']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveF


ACV-MRP results:
ACV-MRP Point estimate: 6860985.051280151
HF-only point estimate: 6856478.716666667
CI: [0.0, 6892977.578114022]
Estimated control variate coefficient: 1.0123087574868095
Estimated sample correlation: 0.989779337102517
Variance reduction factor (spending additional conputation on low-fidelity replications): 39.730667607466636


In [3]:
# STEP 5 - Compute the true finite-population optimality gap in the HF model
true_gap_evaluator = TrueOptimalityGapEvaluator(
    model=hf_model,
    solver_name="gurobi_direct",
    solver_options=None,
)

true_gap_results = true_gap_evaluator.compute_true_gap(xhat=xhat)

print("\nTrue finite-population HF quantities:")
print(f"True optimal value: {true_gap_results['true_optimal_value']}")
print(f"xhat true value: {true_gap_results['xhat_true_value']}")
print(f"True optimality gap: {true_gap_results['true_gap']}")

INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - START
INFO - Using single_bundle scheme (extensive form solve).
INFO - 
INFO - ----------------------------------------------------------------------
INFO - ExtensiveFormSolver - STOP



True finite-population HF quantities:
True optimal value: 9606599.999999875
xhat true value: 16475151.599999784
True optimality gap: 6868551.599999908
